# Notebook 01: Tensors and the Computational Graph

## Why This Matters

In production ML pipelines, silent shape bugs are among the most common causes of wasted GPU hours and subtle model regressions. A model that silently broadcasts a (batch, 1) tensor against a (batch, seq_len) target produces no error—just wrong gradients and a loss curve that looks "fine" for the first few hundred steps. Understanding that a PyTorch tensor is not just a multidimensional array—but a node in a differentiable computation graph with a device placement contract—is the mental model that separates engineers who debug fast from those who waste days.

## Background

PyTorch tensors extend NumPy's ndarray concept with two critical additions:

1. **Gradient tracking** — Every tensor can carry a `grad_fn` pointer back to the operation that created it, forming a directed acyclic graph (DAG) that autograd traverses during `backward()`.
2. **Device placement** — Tensors live on a specific device (CPU, CUDA, MPS). Operations between tensors on different devices raise immediately. This is a feature: you always know where your data is.

The computational graph is built *dynamically* during the forward pass (define-by-run), which means Python control flow is first-class—loops, conditionals, recursion all work naturally.

In [ ]:
%pip install -q torch numpy matplotlib

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {device}")
print(f"torch version: {torch.__version__}")
print("Ready.")

## 1. Tensor Creation and the Non-Array Properties

NumPy arrays are memory buffers with a dtype and shape. PyTorch tensors add `requires_grad`, `grad_fn`, `device`, and `is_leaf`. These aren't cosmetic—they control whether and how the tensor participates in backprop.

**Mental model**: think of `requires_grad=True` as opting a tensor into the autodiff system. Leaf tensors with `requires_grad=True` are the only ones whose `.grad` attribute is populated after `backward()`. Everything else is just a node in the DAG.

In [ ]:
# --- creation methods and their differences ---
t_zeros   = torch.zeros(3, 4)
t_ones    = torch.ones(3, 4)
t_rand    = torch.rand(3, 4)          # uniform [0,1)
t_randn   = torch.randn(3, 4)         # standard normal
t_arange  = torch.arange(0, 12, dtype=torch.float32).reshape(3, 4)
t_linspace = torch.linspace(0, 1, 12).reshape(3, 4)

# from numpy (zero-copy when possible)
arr = np.random.randn(3, 4).astype(np.float32)
t_from_np = torch.from_numpy(arr)     # shares memory!
t_tensor  = torch.tensor(arr)         # always copies

print("t_zeros:   ", t_zeros.shape, t_zeros.dtype, t_zeros.device)
print("t_randn:   ", t_randn.shape,  t_randn.dtype,  t_randn.device)
print("from_numpy shares memory:", np.shares_memory(arr, t_from_np.numpy()))
print("tensor()   shares memory:", np.shares_memory(arr, t_tensor.numpy()))

## 2. Shapes, Dtypes, and Device Placement

Three attributes govern tensor compatibility: `shape`, `dtype`, and `device`. A mismatch on any of the three raises immediately at the operation site—before any silent numerical corruption can occur. When moving to GPU/MPS always use `.to(device)` rather than `.cuda()` to keep your code device-agnostic.

In [ ]:
# dtype promotion rules
x_f32 = torch.tensor([1.0, 2.0], dtype=torch.float32)
x_f64 = torch.tensor([1.0, 2.0], dtype=torch.float64)
result = x_f32 + x_f64   # promotes to float64
print(f"f32 + f64 → {result.dtype}")

# device placement
t_cpu = torch.randn(4, device="cpu")
t_dev = torch.randn(4, device=device)
try:
    _ = t_cpu + t_dev
except RuntimeError as e:
    print(f"Cross-device error (expected): {e}")

# moving tensors
t_on_device = t_cpu.to(device)
print(f"After .to(device): {t_on_device.device}")

# common dtypes and memory cost per element
for dt in [torch.float16, torch.bfloat16, torch.float32, torch.float64, torch.int32]:
    nbytes = torch.zeros(1, dtype=dt).element_size()
    print(f"  {str(dt):25s}  {nbytes} bytes/element")

## 3. Broadcasting — Rules and Traps

Broadcasting follows NumPy semantics but the consequences are more severe in PyTorch: a wrong broadcast can silently produce the correct loss on toy data but fail catastrophically at full batch size. The rule: dimensions are aligned from the right, and a dimension of size 1 (or absent) is stretched to match the other.

**Common trap**: `(batch, 1)` + `(1, classes)` = `(batch, classes)`. Looks like element-wise addition but creates a full matrix. This is fine if intended; catastrophic if you meant a simple per-sample bias.

In [ ]:
# explicit broadcasting demonstration
a = torch.tensor([[1.], [2.], [3.]])   # shape (3, 1)
b = torch.tensor([[10., 20., 30.]])    # shape (1, 3)
c = a + b                              # broadcasts to (3, 3)
print("a shape:", a.shape)
print("b shape:", b.shape)
print("a + b shape:", c.shape)
print(c)

# the dangerous batch case
logits = torch.randn(32, 10)      # (batch, classes)
bias   = torch.randn(10)          # (classes,) — correct
bias_wrong = torch.randn(32, 1)   # (batch, 1) — wrong but no error!
ok_result   = logits + bias        # (32, 10)
wrong_result = logits + bias_wrong # also (32, 10) — silently wrong intent
print(f"\nCorrect add: {ok_result.shape}")
print(f"Dangerous add (no error!): {wrong_result.shape}")

# torch.broadcast_shapes to check intent before running
shape = torch.broadcast_shapes((32, 10), (10,))
print(f"broadcast_shapes((32,10),(10,)) = {shape}")

## 4. requires_grad, grad_fn, and the Computation Graph

When you perform operations on tensors with `requires_grad=True`, PyTorch records those operations in a graph. Each output tensor holds a `grad_fn` pointing to the function that created it. This chain is the graph autograd walks backward.

**Leaf tensors** are created directly by the user (not as the output of an operation). Only leaf tensors accumulate `.grad`. Intermediate nodes discard their gradients by default (memory saving). `retain_graph=True` is needed only when you call `backward()` more than once on the same graph.

In [ ]:
# build a simple computation graph
x = torch.tensor([2.0, 3.0], requires_grad=True)  # leaf
w = torch.tensor([0.5, -1.0], requires_grad=True) # leaf

# forward pass
y = (x * w).sum()    # y = x0*w0 + x1*w1 = 1.0 - 3.0 = -2.0

print(f"x.is_leaf: {x.is_leaf}")
print(f"y.is_leaf: {y.is_leaf}")
print(f"y.grad_fn: {y.grad_fn}")  # SumBackward

# backward pass
y.backward()

# analytical: dy/dx_i = w_i,  dy/dw_i = x_i
print(f"\nx.grad (should be w={w.data}): {x.grad}")
print(f"w.grad (should be x={x.data}): {w.grad}")

# visualise graph depth
def print_graph(node, depth=0, visited=set()):
    if node is None or node in visited:
        return
    visited.add(node)
    print("  " * depth + str(node.__class__.__name__))
    for child, _ in node.next_functions:
        print_graph(child, depth+1, visited)

print("\nGraph (leaf AccumulateGrad = parameter):")
print_graph(y.grad_fn)

## 5. detach() vs torch.no_grad() — When to Use Each

This is one of the most common interview questions and also a real production gotcha.

- `tensor.detach()` creates a new tensor that shares storage but is **removed from the graph**. The original graph is unaffected. Use this when you want to pass an intermediate activation to a loss or metric without propagating gradients back through it (e.g., stop-gradient in contrastive learning, target network in RL).
- `torch.no_grad()` context manager **disables gradient tracking entirely** for all operations within the block. Memory-efficient for inference loops. Does not create a new tensor.
- **Common mistake**: using `detach()` in a training loop when you meant `no_grad()`, causing all-zero gradients to the encoder.

In [ ]:
x = torch.randn(4, requires_grad=True)
h = x * 2 + 1

# detach: h_d has no grad_fn, shares data
h_d = h.detach()
print(f"h.requires_grad:   {h.requires_grad}")
print(f"h_d.requires_grad: {h_d.requires_grad}")
print(f"h_d.grad_fn:       {h_d.grad_fn}")      # None
print(f"same storage:      {h_d.data_ptr() == h.data_ptr()}")

# no_grad: no graph overhead inside the block
with torch.no_grad():
    y = x * 2 + 1
    print(f"\nUnder no_grad: y.requires_grad = {y.requires_grad}")

# typical inference pattern (memory ~halved vs training)
def inference(model_fn, inputs):
    with torch.no_grad():
        return model_fn(inputs)

# stop-gradient pattern (contrastive learning / target networks)
online   = torch.randn(8, 128, requires_grad=True)
target   = online.detach()   # target network: no grads flow back
loss_val = torch.nn.functional.mse_loss(online, target)
loss_val.backward()
print(f"\nStop-grad pattern: online.grad norm = {online.grad.norm():.4f}")

## 6. Common Shape Bugs and How to Debug Them

Shape bugs are the #1 cause of silent failures in PyTorch code. The following patterns appear constantly in production code.

In [ ]:
import traceback

# Bug 1: Forgetting batch dimension
single_image = torch.randn(3, 32, 32)   # C, H, W
conv = torch.nn.Conv2d(3, 16, 3, padding=1)
try:
    _ = conv(single_image)
except RuntimeError as e:
    print(f"Bug 1 (expected): {e}")
    print("Fix: add batch dim with .unsqueeze(0)")
    out = conv(single_image.unsqueeze(0))
    print(f"  Fixed shape: {out.shape}\n")

# Bug 2: squeeze() removing batch dim when batch_size=1
t = torch.randn(1, 10)
print(f"Bug 2: t.squeeze() when batch=1 → {t.squeeze().shape}")  # becomes (10,)
print(f"  Safe fix: t.squeeze(1) → {t.squeeze(1).shape}\n")

# Bug 3: view vs reshape (non-contiguous)
t = torch.randn(4, 6)
t_T = t.T              # transpose creates non-contiguous tensor
try:
    _ = t_T.view(6, 4)
except RuntimeError as e:
    print(f"Bug 3 (expected): {e}")
    print(f"  Fix: t_T.contiguous().view(6,4) → {t_T.contiguous().view(6,4).shape}\n")
    print(f"  Or:  t_T.reshape(6,4)           → {t_T.reshape(6,4).shape}\n")

# Debugging tools
t = torch.randn(2, 3, 4)
print(f"Shape:       {t.shape}")
print(f"ndim:        {t.ndim}")
print(f"numel:       {t.numel()}")
print(f"is_contiguous: {t.is_contiguous()}")
print(f"stride:      {t.stride()}")

## Summary

### Key Concepts

| Concept | Key Point |
|---------|-----------|
| Tensor vs ndarray | Tensors add grad tracking, device placement, and graph membership |
| requires_grad | Opting a leaf tensor into the autodiff DAG |
| grad_fn | Pointer from output tensor back to its creating op; forms the graph |
| Broadcasting | Right-align dims; size-1 stretches — silent bugs if unintended |
| detach() | Remove a tensor from the graph; shares storage; other grads unaffected |
| no_grad() | Disable graph building entirely; use for inference; saves memory |
| view vs reshape | view requires contiguous; reshape is always safe but may copy |
| .to(device) | Device-agnostic; prefer over .cuda() |

### Common Pitfalls
- Using `from_numpy` without realising NumPy and PyTorch share memory — mutations bleed across
- Squeezing without specifying a dimension removes batch dim when batch_size=1
- Broadcasting `(batch, 1)` with `(batch, classes)` produces a full matrix silently
- Calling `.grad` on a non-leaf tensor returns `None` — you must `retain_grad()` explicitly
- Using `detach()` where `no_grad()` was intended, causing encoder to receive zero gradients

### Next: Notebook 02 — Autograd Internals